# Assignment 1: TensorFlow/Keras setup in Colab + preprocessing
Install and configure TensorFlow/Keras, then do preprocessing, normalization, train-test split and visualization on a sample dataset (Wine, from scikit-learn).

## 1. Install and verify TensorFlow/Keras

In [ ]:
# Colab already ships with TensorFlow. Uncomment to force an install/upgrade.
# !pip install -q tensorflow
import tensorflow as tf
from tensorflow import keras
print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)
print("GPUs      :", tf.config.list_physical_devices("GPU"))
# Runtime > Change runtime type > T4 GPU to enable the GPU

In [ ]:
# quick sanity test: tiny model builds and runs
m = keras.Sequential([keras.layers.Input((4,)), keras.layers.Dense(8, activation="relu"), keras.layers.Dense(1)])
print(m(tf.random.normal((2, 4))))

## 2. Load the dataset

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
df = wine.frame.copy()
df["target_name"] = df["target"].map(dict(enumerate(wine.target_names)))
print(df.shape)
df.head()

## 3. Explore the data

In [ ]:
df.info()
print()
print(df.describe().T)
print("\nClass counts:\n", df["target_name"].value_counts())

## 4. Data preprocessing
Check missing values / duplicates, then handle them. Wine has none, so we also inject a few NaNs to demonstrate imputation.

In [ ]:
from sklearn.impute import SimpleImputer

print("Missing values:", df.isna().sum().sum(), "| Duplicates:", df.duplicated().sum())

# demo: inject 1% missing values to show the imputation step working
rng = np.random.default_rng(42)
demo = df.drop(columns=["target", "target_name"]).copy()
mask = rng.random(demo.shape) < 0.01
demo = demo.mask(mask)
print("After injecting NaNs:", demo.isna().sum().sum())

imputer = SimpleImputer(strategy="median")
clean = pd.DataFrame(imputer.fit_transform(demo), columns=demo.columns)
print("After imputation   :", clean.isna().sum().sum())

# for the rest of the assignment use the original (complete) features
X = df.drop(columns=["target", "target_name"])
y = df["target"]

## 5. Train-test split
Stratified so every class keeps the same proportion in both sets. Split **before** scaling to avoid data leakage.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train class dist:", np.bincount(y_train), "Test class dist:", np.bincount(y_test))

## 6. Normalization / standardization
Fit the scaler on training data only, then apply to test data.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

std = StandardScaler().fit(X_train)
mm = MinMaxScaler().fit(X_train)

X_train_std, X_test_std = std.transform(X_train), std.transform(X_test)
X_train_mm, X_test_mm = mm.transform(X_train), mm.transform(X_test)

summary = pd.DataFrame({
    "raw mean": X_train.mean().values[:5],
    "std mean": X_train_std.mean(axis=0)[:5],
    "std std": X_train_std.std(axis=0)[:5],
    "minmax min": X_train_mm.min(axis=0)[:5],
    "minmax max": X_train_mm.max(axis=0)[:5],
}, index=X.columns[:5]).round(3)
summary

## 7. Visualization

In [ ]:
# Class distribution + feature histograms
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="target_name", data=df, ax=ax[0]); ax[0].set_title("Class distribution")
df["alcohol"].plot(kind="hist", bins=20, ax=ax[1], title="Alcohol distribution")
plt.tight_layout(); plt.show()

In [ ]:
# Before vs after scaling (boxplots)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=X_train, orient="h", ax=ax[0]); ax[0].set_title("Raw features")
sns.boxplot(data=pd.DataFrame(X_train_std, columns=X.columns), orient="h", ax=ax[1]); ax[1].set_title("After StandardScaler")
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Feature correlation"); plt.show()

In [ ]:
# Pairplot of a few features coloured by class
sns.pairplot(df[["alcohol", "flavanoids", "color_intensity", "proline", "target_name"]], hue="target_name", corner=True)
plt.show()

## 8. Convert to tf.data pipeline (ready for Keras)

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train_std.astype("float32"), y_train.values)).shuffle(256, seed=42).batch(32).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test_std.astype("float32"), y_test.values)).batch(32).prefetch(tf.data.AUTOTUNE)
for xb, yb in train_ds.take(1):
    print("batch X:", xb.shape, "batch y:", yb.shape)

## Conclusion
TensorFlow/Keras was installed and verified, the Wine dataset was explored, cleaned, split (stratified 80/20), standardized/min-max scaled without leakage, visualized, and wrapped in a `tf.data` pipeline.